## Dijkstra's Algorithm Using a Manually Implemented Binary Min-Heap


### Heap Data Structure

A **heap** is a tree-based data structure commonly used to implement a **priority queue**.

A priority queue differs from an ordinary queue:

- In an ordinary queue, elements are removed in first-in, first-out order.
- In a priority queue, elements are removed according to their priority.

In Dijkstra's algorithm, the priority of a vertex is its current tentative distance from the source. Therefore, the algorithm repeatedly needs to remove the vertex having the smallest tentative distance.

### Binary Heap

A **binary heap** is a binary tree satisfying two properties.

#### Shape Property

A binary heap is a **complete binary tree**. Every level is completely filled except possibly the last level, and the nodes in the last level are filled from left to right.

#### Heap-Order Property

- In a **min-heap**, the value at every node is less than or equal to the values of its children.
- In a **max-heap**, the value at every node is greater than or equal to the values of its children.

Dijkstra's algorithm uses a **min-heap** because it must repeatedly select the vertex having the smallest tentative distance.

For example, the following tree satisfies the min-heap property:

```text
          2
        /   \
       5     4
      / \   /
     9   7 8
```

The smallest element is at the root. However, the entire heap is not necessarily sorted.

### Array Representation

Because a binary heap is complete, it can be stored efficiently in an array. With zero-based indexing, if a node is at index `i`, then:

- left child index: `2*i + 1`
- right child index: `2*i + 2`
- parent index: `(i - 1)//2`

The preceding heap has the array representation:

```text
[2, 5, 4, 9, 7, 8]
```

### Main Operations

#### Insertion

The new element is placed at the end of the array. It is then repeatedly exchanged with its parent while it is smaller than its parent. This process is called **sifting up**.

#### Removal of the Minimum

The root contains the minimum element. To remove it:

1. Save the root element.
2. Move the last element to the root.
3. Remove the last array position.
4. Repeatedly exchange the new root with its smaller child until the min-heap property is restored.

This process is called **sifting down**.

### Time Complexities

For a binary heap containing $n$ elements:

| Operation | Time complexity |
|---|---:|
| Examine the minimum | $O(1)$ |
| Insert an element | $O(\log n)$ |
| Remove the minimum | $O(\log n)$ |

The insertion and removal operations take $O(\log n)$ time because the height of a complete binary tree is $O(\log n)$.


## Manual Binary Min-Heap Implementation

The heap is stored in the list `data`. The methods `_sift_up` and `_sift_down` restore the min-heap property after insertion and removal, respectively.

No built-in heap library is used.


In [1]:
class MinHeap:
    def __init__(self):
        self.data = []

    def is_empty(self):
        return len(self.data) == 0

    def push(self, item):
        """Insert an item into the min-heap."""
        self.data.append(item)
        self._sift_up(len(self.data) - 1)

    def pop(self):
        """Remove and return the smallest item."""
        if self.is_empty():
            raise IndexError("Cannot remove an item from an empty heap.")

        minimum = self.data[0]
        last_item = self.data.pop()

        if not self.is_empty():
            self.data[0] = last_item
            self._sift_down(0)

        return minimum

    def _sift_up(self, index):
        while index > 0:
            parent = (index - 1) // 2

            if self.data[index] >= self.data[parent]:
                break

            self.data[index], self.data[parent] = (
                self.data[parent], self.data[index]
            )
            index = parent

    def _sift_down(self, index):
        size = len(self.data)

        while True:
            left = 2 * index + 1
            right = 2 * index + 2
            smallest = index

            if left < size and self.data[left] < self.data[smallest]:
                smallest = left

            if right < size and self.data[right] < self.data[smallest]:
                smallest = right

            if smallest == index:
                break

            self.data[index], self.data[smallest] = (
                self.data[smallest], self.data[index]
            )
            index = smallest


## Test the Binary Min-Heap

Insert several values and observe the internal array after each insertion. The array need not be completely sorted, but it must satisfy the min-heap property.


In [2]:
heap = MinHeap()
values = [9, 4, 7, 1, 6, 2, 8]

for value in values:
    heap.push(value)
    print(f"After inserting {value}: {heap.data}")


After inserting 9: [9]
After inserting 4: [4, 9]
After inserting 7: [4, 9, 7]
After inserting 1: [1, 4, 7, 9]
After inserting 6: [1, 4, 7, 9, 6]
After inserting 2: [1, 4, 2, 9, 6, 7]
After inserting 8: [1, 4, 2, 9, 6, 7, 8]


Remove the elements one by one. Since this is a min-heap, they must be removed in increasing order.


In [3]:
print("Elements removed from the heap:")

while not heap.is_empty():
    print(heap.pop(), end=" ")


Elements removed from the heap:
1 2 4 6 7 8 9 

## Weighted Graph

The graph is represented using an adjacency list. Each pair `(neighbour, weight)` represents a weighted edge.


In [4]:
graph = {
    'A': [('B', 4), ('C', 2)],
    'B': [('A', 4), ('D', 5)],
    'C': [('A', 2), ('D', 8), ('E', 10)],
    'D': [('B', 5), ('C', 8), ('E', 2)],
    'E': [('C', 10), ('D', 2)]
}

graph


{'A': [('B', 4), ('C', 2)],
 'B': [('A', 4), ('D', 5)],
 'C': [('A', 2), ('D', 8), ('E', 10)],
 'D': [('B', 5), ('C', 8), ('E', 2)],
 'E': [('C', 10), ('D', 2)]}

## Dijkstra's Algorithm Using Min-Heap

For each vertex, the algorithm stores:

- `distance[vertex]`: the currently known shortest distance from the source;
- `parent[vertex]`: the preceding vertex on the currently known shortest path.

Each heap entry has the form `(distance, vertex)`. Tuples are compared from left to right, so an entry with a smaller distance has higher priority.

When a shorter distance is found, a new entry is inserted into the heap. An older entry for the same vertex may remain in the heap. Such an entry is stale and is ignored using the test

```python
if current_distance != distance[u]:
    continue
```


In [5]:
def dijkstra(graph, source):
    distance = {vertex: float('inf') for vertex in graph}
    parent = {vertex: None for vertex in graph}

    distance[source] = 0

    priority_queue = MinHeap()
    priority_queue.push((0, source))

    while not priority_queue.is_empty():
        current_distance, u = priority_queue.pop()

        # Ignore a stale heap entry.
        if current_distance != distance[u]:
            continue

        for v, weight in graph[u]:
            new_distance = current_distance + weight

            if new_distance < distance[v]:
                distance[v] = new_distance
                parent[v] = u
                priority_queue.push((new_distance, v))

    return distance, parent


## Reconstructing a Shortest Path

The `parent` dictionary gives the preceding vertex on a shortest path. Starting from the destination, follow the parent pointers until the source is reached, and then reverse the resulting sequence.


In [6]:
def reconstruct_path(parent, source, destination):
    path = []
    current = destination

    while current is not None:
        path.append(current)

        if current == source:
            return path[::-1]

        current = parent[current]

    return None


## Run Dijkstra's Algorithm


In [7]:
source = 'A'
distance, parent = dijkstra(graph, source)

print("Shortest distances from", source)

for vertex in sorted(graph):
    print(f"{source} -> {vertex}: {distance[vertex]}")


Shortest distances from A
A -> A: 0
A -> B: 4
A -> C: 2
A -> D: 9
A -> E: 11


In [8]:
destination = 'E'
path = reconstruct_path(parent, source, destination)

if path is None:
    print(f"No path exists from {source} to {destination}.")
else:
    print("Shortest path:", " -> ".join(path))
    print("Path cost:", distance[destination])


Shortest path: A -> B -> D -> E
Path cost: 11


## Display the Shortest Path to Every Vertex


In [9]:
for destination in sorted(graph):
    path = reconstruct_path(parent, source, destination)

    if path is None:
        print(f"{destination}: unreachable")
    else:
        print(
            f"{destination}: {' -> '.join(path)}, "
            f"cost = {distance[destination]}"
        )


A: A, cost = 0
B: A -> B, cost = 4
C: A -> C, cost = 2
D: A -> B -> D, cost = 9
E: A -> B -> D -> E, cost = 11


## Exercises

1. Change the source vertex from `A` to `C` and display all shortest paths.
2. Add a new vertex and at least two weighted edges to the graph. Run the algorithm again and justify why the output is correct. 
3. Modify the algorithm to record the order in which vertices are settled.


### Note:
Applications of Dijkstra's algorithm include:

- route planning and navigation,
- packet routing in computer networks,
- transportation and logistics,
- robotics, etc.

In [10]:
# Exercise 1
source = 'C'

distance, parent = dijkstra(graph, source)

print("Shortest distances from", source)

for destination in sorted(graph):
    path = reconstruct_path(parent, source, destination)

    if path is None:
        print(f"{destination}: unreachable")
    else:
        print(
            f"{destination}: {' -> '.join(path)}, "
            f"cost = {distance[destination]}"
        )

Shortest distances from C
A: C -> A, cost = 2
B: C -> A -> B, cost = 6
C: C, cost = 0
D: C -> D, cost = 8
E: C -> E, cost = 10


In [11]:
# Exercise 2

graph['F'] = [
    ('D', 3),
    ('E', 4)
]

graph['D'].append(('F', 3))
graph['E'].append(('F', 4))

source = 'A'

distance, parent = dijkstra(graph, source)

for destination in sorted(graph):
    path = reconstruct_path(parent, source, destination)

    if path is None:
        print(f"{destination}: unreachable")
    else:
        print(
            f"{destination}: {' -> '.join(path)}, "
            f"cost = {distance[destination]}"
        )

A: A, cost = 0
B: A -> B, cost = 4
C: A -> C, cost = 2
D: A -> B -> D, cost = 9
E: A -> B -> D -> E, cost = 11
F: A -> B -> D -> F, cost = 12


In [15]:
# Exercise 3

def dijkstra(graph, source):
    distance = {vertex: float('inf') for vertex in graph}
    parent = {vertex: None for vertex in graph}

    distance[source] = 0

    priority_queue = MinHeap()
    priority_queue.push((0, source))

    settled_order = []

    while not priority_queue.is_empty():
        current_distance, u = priority_queue.pop()

        # Ignore a stale heap entry.
        if current_distance != distance[u]:
            continue

        settled_order.append(u)

        for v, weight in graph[u]:
            new_distance = current_distance + weight

            if new_distance < distance[v]:
                distance[v] = new_distance
                parent[v] = u
                priority_queue.push((new_distance, v))

    return distance, parent, settled_order



#MAIN
source = 'A'

distance, parent, settled_order = dijkstra(graph, source)

print("Order in which vertices were settled:")
print(" -> ".join(settled_order))

print("\nShortest paths:")

for destination in sorted(graph):
    path = reconstruct_path(parent, source, destination)

    if path is None:
        print(f"{destination}: unreachable")
    else:
        print(
            f"{destination}: {' -> '.join(path)}, "
            f"cost = {distance[destination]}"
        )

Order in which vertices were settled:
A -> C -> B -> D -> E -> F

Shortest paths:
A: A, cost = 0
B: A -> B, cost = 4
C: A -> C, cost = 2
D: A -> B -> D, cost = 9
E: A -> B -> D -> E, cost = 11
F: A -> B -> D -> F, cost = 12
